In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')

print(df.shape)
df.head()



(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df.columns.tolist()


['customerID',
 'gender',
 'SeniorCitizen',
 'Partner',
 'Dependents',
 'tenure',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'MonthlyCharges',
 'TotalCharges',
 'Churn']

In [ ]:
# Q1: Data Quality Report
def data_quality_report(df):
    report = pd.DataFrame({
        'data_type': df.dtypes,
        'null_count': df.isnull().sum(),
        'null_percentage': round(df.isnull().sum() / len(df) * 100, 2),
        'distinct_count': df.nunique(),
        'duplicate_count': df.duplicated().sum(),
    })

    report['min'] = df.min(numeric_only=True)
    report['max'] = df.max(numeric_only=True)

    return report

data_quality_report(df)

In [8]:
# Q2: TotalCharges is stored as object — find invalid values, convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

invalid_count = df['TotalCharges'].isnull().sum()
print(f'Invalid values found and converted to NaN: {invalid_count}')

df[df['TotalCharges'].isnull()]

df['TotalCharges'] = df['TotalCharges'].fillna(0)

print(f'Remaining nulls in TotalCharges: {df["TotalCharges"].isnull().sum()}')
print(f'TotalCharges dtype: {df["TotalCharges"].dtype}')

Invalid values found and converted to NaN: 0
Remaining nulls in TotalCharges: 0
TotalCharges dtype: float64


In [ ]:
# Q3: Function that categorizes columns into groups
def categorize_columns(df):
    numerical = [] 
    categorical = []
    binary = []
    identifier = []

    for col in df.columns:
        unique_vals = df[col].nunique()

        if unique_vals == len(df):
            identifier.append(col)
        elif unique_vals == 2:
            binary.append(col)
        elif df[col].dtype in ['int64', 'float64']:
            numerical.append(col)
        else:
            categorical.append(col)

    return {
        'Identifier': identifier,
        'Binary': binary,
        'Numerical': numerical,
        'Categorical': categorical
    }

categorize_columns(df)

In [9]:
# Q4: Identify customers where TotalCharges differs from MonthlyCharges x tenure by more than 10%
df['expected_total'] = df['MonthlyCharges'] * df['tenure']
df['diff_percentage'] = abs(df['TotalCharges'] - df['expected_total']) / (df['expected_total'] + 1) * 100

suspicious = df[df['diff_percentage'] > 10]

print(f'Customers with >10% discrepancy: {len(suspicious)}')
suspicious[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'expected_total', 'diff_percentage']].head(10)

Customers with >10% discrepancy: 377


,customerID,tenure,MonthlyCharges,TotalCharges,expected_total,diff_percentage
21,1680-VDCWW,12,19.80,202.25,237.60,14.815591
42,9867-JCZSP,17,20.75,418.25,352.75,18.515901
47,7760-OYPDY,2,80.65,144.15,161.30,10.566852
69,7410-OIEDU,10,79.85,887.35,798.50,11.113196
77,5590-ZSKRV,8,54.65,482.25,437.20,10.280694
105,6180-YBIQI,5,24.30,100.20,121.50,17.387755
124,7219-TLZHO,4,20.85,62.90,83.40,24.289100
171,1875-QIVME,2,104.40,242.80,208.80,16.205910
223,0742-MOABM,4,50.05,179.35,200.20,10.362823
351,5160-UXJED,17,44.60,681.40,758.20,10.115911
